Imports

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

Rutas

In [10]:
BASE           = Path("../../")
PRED_DIR       = BASE / "models" / "predictions"
DASHBOARD_DATA = BASE / "dashboard" / "data"
DASHBOARD_DATA.mkdir(parents=True, exist_ok=True)
print("Predictions dir:", PRED_DIR)
print("Dashboard data:", DASHBOARD_DATA)

Predictions dir: ..\..\models\predictions
Dashboard data: ..\..\dashboard\data


Cargar todos los ficheros de predicciones

In [11]:
pred_files = list(PRED_DIR.glob("*_predictions.parquet"))
print("Ficheros encontrados:", len(pred_files))
pred_files

Ficheros encontrados: 5


[WindowsPath('../../models/predictions/arima_predictions.parquet'),
 WindowsPath('../../models/predictions/random_forest_predictions.parquet'),
 WindowsPath('../../models/predictions/sarimax_predictions.parquet'),
 WindowsPath('../../models/predictions/sarima_predictions.parquet'),
 WindowsPath('../../models/predictions/xgb_predictions.parquet')]

Unificar todas las predicciones

In [12]:
dfs = []
for file in pred_files:
    df = pd.read_parquet(file)
    df["modelo"] = df["modelo"].astype(str)
    df["hotel"]  = df["hotel"].astype(str)
    df["fecha"]  = pd.to_datetime(df["fecha"])
    dfs.append(df)

df_pred = pd.concat(dfs, ignore_index=True)
df_pred = df_pred.sort_values(["hotel", "modelo", "fecha"]).reset_index(drop=True)
df_pred.head()

,fecha,hotel,modelo,y_real,y_pred
0,2025-01-01,HOTEL_1,ARIMA,0.734043,0.619440
1,2025-01-02,HOTEL_1,ARIMA,0.702128,0.581788
2,2025-01-03,HOTEL_1,ARIMA,0.691489,0.558703
3,2025-01-04,HOTEL_1,ARIMA,0.521277,0.544549
4,2025-01-05,HOTEL_1,ARIMA,0.489362,0.535872


Calcular errores para evaluación y dashboard

In [13]:
df_pred["error_abs"] = (df_pred["y_pred"] - df_pred["y_real"]).abs()
df_pred["error_pct"] = df_pred["error_abs"] / df_pred["y_real"].replace(0, np.nan)
df_pred

,fecha,hotel,modelo,y_real,y_pred,error_abs,error_pct
0,2025-01-01,HOTEL_1,ARIMA,0.734043,0.619440,0.114603,0.156126
1,2025-01-02,HOTEL_1,ARIMA,0.702128,0.581788,0.120340,0.171393
2,2025-01-03,HOTEL_1,ARIMA,0.691489,0.558703,0.132786,0.192030
3,2025-01-04,HOTEL_1,ARIMA,0.521277,0.544549,0.023273,0.044646
4,2025-01-05,HOTEL_1,ARIMA,0.489362,0.535872,0.046510,0.095042
...,...,...,...,...,...,...,...
4235,2025-12-19,HOTEL_3,XGBoost,0.681275,0.673672,0.007602,0.011159
4236,2025-12-20,HOTEL_3,XGBoost,0.713147,0.695552,0.017596,0.024673
4237,2025-12-21,HOTEL_3,XGBoost,0.685259,0.669532,0.015727,0.022950
4238,2025-12-22,HOTEL_3,XGBoost,0.685259,0.677557,0.007702,0.011240


Añadir columnas temporales para segmentación

In [14]:
df_pred["year"]  = df_pred["fecha"].dt.year
df_pred["month"] = df_pred["fecha"].dt.month
df_pred["dow"]   = df_pred["fecha"].dt.dayofweek       # 0 = lunes, 6 = domingo
df_pred["week"]  = df_pred["fecha"].dt.isocalendar().week.astype(int)

Guardar dataset final

In [15]:
output_file = DASHBOARD_DATA / "predictions_dataset.parquet"
df_pred.to_parquet(output_file, index=False)
print("✅ predictions_dataset.parquet guardado en:", output_file)

✅ predictions_dataset.parquet guardado en: ..\..\dashboard\data\predictions_dataset.parquet


Resumen rápido para visualizar

In [16]:
(
    df_pred.groupby(["hotel", "modelo"])["error_abs"]
           .mean()
           .to_frame("MAE_estimado")
           .sort_values("MAE_estimado")
)

MAE_estimado
hotel   modelo                    
HOTEL_3 XGBoost           0.035688
        RandomForest      0.036515
HOTEL_2 XGBoost           0.043035
HOTEL_1 XGBoost           0.052480
HOTEL_3 SARIMAX           0.052725
HOTEL_2 RandomForest      0.067398
HOTEL_1 RandomForest      0.069182
        SARIMAX           0.088223
HOTEL_3 ARIMA             0.095105
        SARIMA            0.097969
HOTEL_1 ARIMA             0.118347
        SARIMA            0.122953
HOTEL_2 SARIMA            0.167298
        ARIMA             0.167308
        SARIMAX           0.264047